# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayaahmed571/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule

I will prioritize pages for refresh when they are stale and still have meaningful search visibility.

The baseline score gives higher priority to pages that have been updated a long time ago and still receive search impressions. This is a simple decision-support baseline, not a claim about Google's ranking algorithm.

### Reason codes

- STALE_WITH_VISIBILITY: the page has been stale for at least 180 days and has at least 500 impressions in the observed 90-day window.
- STALE_LOW_VISIBILITY: the page has been stale for at least 180 days but has fewer than 500 impressions.
- FRESH_WITH_VISIBILITY: the page is not yet stale but has at least 500 impressions.
- OTHER: none of the above conditions apply.

The action for STALE_WITH_VISIBILITY is "Review for refresh". The other groups are lower priority for this baseline.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

df["baseline_score"] = (
    (df["days_since_last_update"] >= 180).astype(int) * 2
    + (df["impressions_90d"] >= 500).astype(int)
)

df["reason_code"] = np.select(
    [
        (df["days_since_last_update"] >= 180) &
        (df["impressions_90d"] >= 500),

        (df["days_since_last_update"] >= 180) &
        (df["impressions_90d"] < 500),

        (df["days_since_last_update"] < 180) &
        (df["impressions_90d"] >= 500)
    ],
    [
        "STALE_WITH_VISIBILITY",
        "STALE_LOW_VISIBILITY",
        "FRESH_WITH_VISIBILITY"
    ],
    default="OTHER"
)

df["action"] = np.where(
    df["reason_code"] == "STALE_WITH_VISIBILITY",
    "Review for refresh",
    "Lower priority"
)

print(df[["baseline_score", "reason_code", "action"]].head())

(30000, 44)
   baseline_score            reason_code          action
0               1  FRESH_WITH_VISIBILITY  Lower priority
1               1  FRESH_WITH_VISIBILITY  Lower priority
2               1  FRESH_WITH_VISIBILITY  Lower priority
3               1  FRESH_WITH_VISIBILITY  Lower priority
4               1  FRESH_WITH_VISIBILITY  Lower priority


In [ ]:
queue = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

import os
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(f"Saved {len(queue)} rows to work/outputs/baseline_action_score.csv")

queue.head(20)

Saved 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,0.84,24.11,0.0,excellent,striking,down,-85.6,3,STALE_WITH_VISIBILITY,Review for refresh
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,3.66,42.99,0.0,excellent,page_3_5,down,-81.5,3,STALE_WITH_VISIBILITY,Review for refresh
7021,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.0,good,page_3_5,down,-74.7,3,STALE_WITH_VISIBILITY,Review for refresh
21268,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3478.0,21948.0,...,5.13,41.76,0.0,good,striking,down,-51.8,3,STALE_WITH_VISIBILITY,Review for refresh
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,0.00,40.00,0.0,good,page_3_5,down,-89.1,3,STALE_WITH_VISIBILITY,Review for refresh
12045,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4758.0,30070.0,...,0.00,40.00,0.0,good,striking,down,-62.8,3,STALE_WITH_VISIBILITY,Review for refresh
698,content_b16bd7307b39,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4329.0,27844.0,...,0.00,25.00,0.0,good,page_3_5,down,-69.7,3,STALE_WITH_VISIBILITY,Review for refresh
5327,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,2.38,38.64,0.0,good,striking,down,-52.2,3,STALE_WITH_VISIBILITY,Review for refresh
26810,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4486.0,29333.0,...,25.00,33.33,0.0,good,page_3_5,down,-74.4,3,STALE_WITH_VISIBILITY,Review for refresh
20837,content_928af3e22c80,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3118.0,20396.0,...,0.00,0.00,0.0,moderate,striking,down,-45.7,3,STALE_WITH_VISIBILITY,Review for refresh


## 3. Top-20 review

I reviewed the top 20 pages selected by the baseline rule. The rule mainly prioritizes pages that have not been updated for at least 180 days and have at least 500 impressions in the last 90 days.

The confidence notes below are intentionally cautious because this is a simple baseline using only freshness and visibility. A page matching the rule does not necessarily mean that refreshing it is the best action.

In [ ]:
top20 = queue.head(20).copy()

top20[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]]

,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr
16751,content_cf56e2e2e282,3,STALE_WITH_VISIBILITY,Review for refresh,194,61678,19.7,0.15
16514,content_7368877ea310,3,STALE_WITH_VISIBILITY,Review for refresh,194,59472,24.8,0.13
7021,content_1bfaa38ff26c,3,STALE_WITH_VISIBILITY,Review for refresh,194,25715,22.2,0.23
21268,content_0a91db491d14,3,STALE_WITH_VISIBILITY,Review for refresh,193,13299,10.5,0.49
11489,content_5feee3994adb,3,STALE_WITH_VISIBILITY,Review for refresh,194,7812,39.0,0.01
12045,content_c2d929d83eaa,3,STALE_WITH_VISIBILITY,Review for refresh,193,7558,17.9,0.20
698,content_b16bd7307b39,3,STALE_WITH_VISIBILITY,Review for refresh,194,4590,31.0,0.00
5327,content_fe16a55cd13d,3,STALE_WITH_VISIBILITY,Review for refresh,194,4556,16.4,0.33
26810,content_ecb6215e79fd,3,STALE_WITH_VISIBILITY,Review for refresh,194,4429,25.3,0.38
20837,content_928af3e22c80,3,STALE_WITH_VISIBILITY,Review for refresh,193,1697,15.8,0.12


| Rank | Action | Reason code | Confidence note | What would make it wrong |
|---|---|---|---|---|
| 1 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; very high visibility (61,678 impressions) and stale for 194 days. | It could still be performing adequately, so a refresh may not be the best use of effort. |
| 2 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; very high visibility (59,472 impressions) and stale for 194 days. | High impressions alone do not mean the content needs refreshing. |
| 3 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; high visibility (25,715 impressions) and stale for 194 days. | The page may already satisfy its purpose despite being old. |
| 4 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 13,299 impressions and 193 days since update. | The page may not have a clear refresh opportunity. |
| 5 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 7,812 impressions but very low CTR (0.01). | Low CTR may have causes other than content freshness. |
| 6 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 7,558 impressions and 193 days since update. | The page may still be useful without a refresh. |
| 7 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 4,590 impressions and 194 days since update. | Zero observed CTR may require investigation rather than automatically refreshing the content. |
| 8 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 4,556 impressions and 194 days since update. | The page may not benefit enough from a refresh to justify the effort. |
| 9 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 4,429 impressions and 194 days since update. | Its CTR and position do not by themselves establish that a refresh is needed. |
| 10 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 1,697 impressions and 193 days since update. | Visibility is much lower than the highest-ranked pages, so the opportunity may be smaller. |
| 11 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 1,408 impressions and 183 days since update. | The page ranks relatively well, so changing it could potentially add little value. |
| 12 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 1,316 impressions and 194 days since update. | The page may not have enough opportunity to justify immediate work. |
| 13 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 954 impressions and 301 days since update. | Being very old does not guarantee that updating it will improve outcomes. |
| 14 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 828 impressions and 194 days since update. | Lower visibility means the expected impact may be limited. |
| 15 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 821 impressions and 301 days since update. | The page has a relatively strong position, so a refresh might not be necessary. |
| 16 | Review for refresh | STALE_WITH_VISIBILITY | Moderate; 545 impressions and 183 days since update. | The low visibility may make this a lower-value refresh than higher-visibility pages. |
| 17 | Review for refresh | STALE_WITH_VISIBILITY | Low-to-moderate; only 533 impressions and CTR is 0.00. | Very low visibility and CTR could indicate that refresh is not the right intervention. |
| 18 | Lower priority | STALE_LOW_VISIBILITY | Moderate; stale for 183 days but only 429 impressions. | The page could still have strategic value that impressions do not capture. |
| 19 | Lower priority | STALE_LOW_VISIBILITY | Moderate; stale for 183 days and only 371 impressions. | Low impressions do not necessarily mean the page is unimportant. |
| 20 | Lower priority | STALE_LOW_VISIBILITY | Moderate; stale for 211 days and 345 impressions. | The rule may underestimate pages with low traffic but high strategic value. |

## 4. Weak picks + leakage check

Some picks in the ranked queue are weaker than others. A stale page does not automatically mean that refreshing it is the best action, so the baseline should be treated as decision-support rather than a final recommendation.

For example, a page can be selected because it is stale even when its observed signals do not suggest an urgent action. This shows a limitation of using only freshness and visibility.

### Leakage check

The baseline score uses only:
- `days_since_last_update`
- `impressions_90d`

I did not use `trend_direction`, `trend_pct`, or `is_declining_label` to calculate the score or reason code. I also did not use future-window information.

Therefore, the baseline does not intentionally include label-derived or future information.

In [ ]:
rule_features = [
    "days_since_last_update",
    "impressions_90d"
]

print("Features used by baseline:")
print(rule_features)

print("\nLabel/future-derived columns NOT used:")
print(["trend_direction", "trend_pct", "is_declining_label"])


Features used by baseline:
['days_since_last_update', 'impressions_90d']

Label/future-derived columns NOT used:
['trend_direction', 'trend_pct', 'is_declining_label']


In [ ]:
weak_picks = queue.tail(10)[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]]

weak_picks

,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr
29753,content_a97955b77e18,0,OTHER,Lower priority,8,1,2.0,0.0
29755,content_d557b826e19f,0,OTHER,Lower priority,20,1,9.0,0.0
29773,content_4f11749a1306,0,OTHER,Lower priority,20,1,0.0,0.0
29824,content_cc19609c216e,0,OTHER,Lower priority,20,1,0.0,0.0
29856,content_321383d0e9cf,0,OTHER,Lower priority,20,1,0.0,0.0
29872,content_6bca373d8751,0,OTHER,Lower priority,20,1,0.0,0.0
29885,content_7d12e9e7a4a5,0,OTHER,Lower priority,20,1,0.0,0.0
29895,content_901b40631379,0,OTHER,Lower priority,98,1,15.0,0.0
29992,content_23dce6a656e6,0,OTHER,Lower priority,20,1,0.0,0.0
29995,content_c322796023c8,0,OTHER,Lower priority,20,1,0.0,0.0


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.